# Data Domain UpSet plot

Reads `data/papers.csv`, parses the `Data Domain` column (a comma-separated list of `EHR`, `Bio-med`, `Generated`, `Other` -- where `Other` may have parenthetical detail), and emits R code that builds a `data.frame` of TRUE/FALSE indicators suitable for `ComplexUpset::upset`.

Run the cells, then copy the printed R block into RStudio (or use the `.R` file written next to this notebook).

In [1]:
import re
from pathlib import Path

import pandas as pd

VIZ_ROOT = Path("..").resolve()
HERE = Path(".").resolve()
DATA = VIZ_ROOT / "data" / "papers.csv"

CATEGORIES = ["EHR", "Bio-med", "Generated", "Other"]

In [2]:
df = pd.read_csv(DATA)
df["Data Domain"].value_counts(dropna=False)

Data Domain
Generated                                      26
Bio-Med                                        19
Bio-med                                        17
EHR                                             6
Generated                                       2
Other (Originally written prompts)              2
Bio-Med, Generated                              2
Generated, EHR                                  1
Other                                           1
Generated, EHR\n                                1
Bio-Med\n                                       1
Bio-med, Generated                              1
Other (reddit)                                  1
Bio-Med, EHR                                    1
Other (questions from Facebook)                 1
EHR, Generated                                  1
Other (Originally written prompts), Bio-Med     1
EHR, Bio-med                                    1
Name: count, dtype: int64

In [3]:
# Split the cell on commas that are NOT inside parentheses, so an entry like
#   "Other (Originally written prompts), Bio-Med"
# yields ["Other (Originally written prompts)", "Bio-Med"].
_TOP_LEVEL_COMMA = re.compile(r",(?![^(]*\))")


def normalize(token: str) -> str | None:
    """Map a single token to one of the canonical CATEGORIES (or None)."""
    t = token.strip().strip(".").strip()
    if not t:
        return None
    low = t.lower()
    if low.startswith("other"):
        return "Other"
    if low in {"ehr"}:
        return "EHR"
    if low in {"bio-med", "biomed", "bio med"}:
        return "Bio-med"
    if low in {"generated"}:
        return "Generated"
    return None


def parse_domains(cell: object) -> set[str]:
    if not isinstance(cell, str):
        return set()
    parts = _TOP_LEVEL_COMMA.split(cell)
    return {c for c in (normalize(p) for p in parts) if c}


rows = []
unrecognized = []
for raw in df["Data Domain"]:
    cats = parse_domains(raw)
    if isinstance(raw, str) and not cats:
        unrecognized.append(raw)
    rows.append({c: (c in cats) for c in CATEGORIES})

domain_df = pd.DataFrame(rows, columns=CATEGORIES)
print("Rows:", len(domain_df))
print("Per-category counts:")
print(domain_df.sum())
if unrecognized:
    print("\nUnrecognized entries (left as all-FALSE):")
    for u in unrecognized:
        print(" -", repr(u))

Rows: 85
Per-category counts:
EHR          11
Bio-med      43
Generated    34
Other         6
dtype: int64


In [4]:
domain_df.head(10)

,EHR,Bio-med,Generated,Other
0,True,False,True,False
1,False,False,True,False
2,False,False,False,True
3,False,True,False,True
4,True,False,True,False
5,True,False,False,False
6,False,True,False,False
7,False,True,False,False
8,False,False,True,False
9,False,False,False,True


In [5]:
def to_r_logical(series: pd.Series) -> str:
    return "c(" + ", ".join("TRUE" if v else "FALSE" for v in series) + ")"


def to_r_string_vector(values: list[str]) -> str:
    return "c(" + ", ".join(f'"{v}"' for v in values) + ")"


lines = [
    "library(ComplexUpset)",
    "library(ggplot2)",
    "",
    "data <- data.frame(",
]
col_lines = [f'  "{c}" = {to_r_logical(domain_df[c])}' for c in CATEGORIES]
lines.append(",\n".join(col_lines) + ",")
lines.append("  check.names = FALSE")
lines.append(")")
lines.append("")
lines.append(f"ComplexUpset::upset(data, intersect = {to_r_string_vector(CATEGORIES)})")

r_code = "\n".join(lines)
print(r_code)

library(ComplexUpset)
library(ggplot2)

data <- data.frame(
  "EHR" = c(TRUE, FALSE, FALSE, FALSE, TRUE, TRUE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, TRUE, FALSE, FALSE, FALSE, TRUE, FALSE, TRUE, FALSE, FALSE, FALSE, FALSE, FALSE, TRUE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, TRUE, TRUE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, TRUE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, TRUE),
  "Bio-med" = c(FALSE, FALSE, FALSE, TRUE, FALSE, FALSE, TRUE, TRUE, FALSE, FALSE, FALSE, FALSE, FALSE, TRUE, TRUE, TRUE, TRUE, TRUE, FALSE, FALSE, TRUE, FALSE, TRUE, TRUE, TRUE, TRUE, TRUE, TRUE, TRUE, TRUE, FALSE, TRUE, FALSE, TRUE, TRUE, TRUE, TRUE, TRUE, FALSE, TRUE, TRUE, TRUE, FALSE, FALSE, FALSE, TRUE, FALSE, FALSE, FALSE, TRUE, FALSE

In [6]:
out_path = HERE / "data_domain_upset.R"
out_path.write_text(r_code + "\n")
print("Wrote", out_path)

Wrote /Users/josh/Desktop/harvard/kempner/viz/data_domain/data_domain_upset.R
